<a href="https://colab.research.google.com/github/GuardinTheDev/Is-This-Text-Ai-/blob/Model-E%C4%9Fitimi/ModelE%C4%9Fitimi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

In [ ]:
from datasets import load_dataset, concatenate_datasets

print("Hugging Face Hub'dan 'Yunij/kaggle-comp-daigt' yükleniyor...")
# Orijinal veri setinin sadece 'train' kısmını alıyoruz (Birleştirme yapabilmek için)
raw_dataset = load_dataset('Yunij/kaggle-comp-daigt', split='train')

# =====================================================================
# 💉 YENİ EKLENEN KISIM: RESMİ İNSAN METİNLERİ (WIKIPEDIA) ENJEKSİYONU
# =====================================================================
print("Dengeleme için resmi insan makaleleri (Wikipedia) indiriliyor...")
# Wikipedia makalelerini içeren popüler bir dil veri setini çekiyoruz
wiki_dataset = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')

# Modelin kafasını karıştıracak kısa başlıkları atalım, sadece 50 kelimeden uzun, gerçek makale paragraflarını alalım
wiki_filtered = wiki_dataset.filter(lambda x: len(x['text'].split()) > 50)

# Veri setinden 3000 adet ciddi insan makalesi alıyoruz (Eğer istersen bu sayıyı 5000 falan yapabilirsin)
wiki_samples = wiki_filtered.select(range(10000))

# Orijinal veri setindeki etiket sütununun adını dinamik bulalım ('label' veya 'generated' olabilir)
orijinal_sutunlar = raw_dataset.column_names
label_col = 'label' if 'label' in orijinal_sutunlar else 'generated'
text_col = 'text'

# Wikipedia metinlerini, senin veri setinle aynı formata sokuyoruz ve hepsine İNSAN (0) etiketini basıyoruz
def format_wiki(example):
    return {text_col: example['text'], label_col: 0}

wiki_formatted = wiki_samples.map(format_wiki, remove_columns=wiki_samples.column_names)

# Birleşmede hata çıkmaması için orijinal veri setindeki gereksiz/ekstra sütunları siliyoruz
sutunlari_tut = [text_col, label_col]
raw_dataset = raw_dataset.remove_columns([col for col in orijinal_sutunlar if col not in sutunlari_tut])

# Orijinal Kaggle verisi ile yeni Wikipedia makalelerini BİRLEŞTİRİYORUZ
print("Orijinal veri seti ile yeni resmi makaleler harmanlanıyor...")
combined_dataset = concatenate_datasets([raw_dataset, wiki_formatted])

# Model sırayla hep aynı şeyleri görmesin diye verileri iyice karıştırıyoruz
combined_dataset = combined_dataset.shuffle(seed=42)
# =====================================================================

# Eğitim ve Doğrulama (Validation) olarak bölelim
print("Eğitim ve doğrulama setleri ayrılıyor...")
split_data = combined_dataset.train_test_split(test_size=0.2, seed=42)
dataset = {
    'train': split_data['train'],
    'validation': split_data['test']
}

print("\n✅ Veri Seti Başarıyla Güncellendi, Dengelendi ve Bölümleri Ayarlandı!")
print("Eğitim seti boyutu:", len(dataset['train']))
print("Doğrulama seti boyutu:", len(dataset['validation']))
print("\nÖrnek bir veri (eğitim setinden):", dataset['train'][0])


In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

# Model altyapısını hazır veri setine bağlama adımı
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_fonksiyonu(examples):
    # DİKKAT: padding='max_length' kısmını sildik!
    # Sadece 512'den uzunları kesiyoruz (truncation).
    return tokenizer(examples['text'], truncation=True, max_length=512)

# Hafızadaki yerel veri setimizi tokenlaştırıyoruz
tokenized_datasets = {
    'train': dataset['train'].map(tokenize_fonksiyonu, batched=True),
    'validation': dataset['validation'].map(tokenize_fonksiyonu, batched=True)
}

# 🚀 DİNAMİK PADDING İÇİN DATA COLLATOR (Eğitimi Hızlandırır)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Tokenlaştırma tamamlandı, model eğitimine hazır!")

In [ ]:
import torch
import numpy as np
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

# 1. Cihaz Kontrolü (Colab'da T4 GPU seçiliyse 'cuda' aktif olur, yoksa 'cpu' çalışır)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Eğitim için kullanılan cihaz: {device}")

# 2. Modeli Sınıflandırma İçin Yüklüyoruz
# İnsan (0) ve Yapay Zeka (1) olmak üzere 2 sınıfımız olduğu için num_labels=2 yapıyoruz
model_name = "distilbert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

# 3. Model Başarısını Ölçmek İçin Metrik Fonksiyonu
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='binary')
    return {"accuracy": acc, "f1": f1}

# 4. Eğitim Hiperparametreleri (Ayarları)
training_args = TrainingArguments(
    output_dir="./ai_detector_local_results", # Çıktıların kaydedileceği klasör
    learning_rate=2e-5,                       # Transformer modelleri için ideal öğrenme oranı
    per_device_train_batch_size=16,            # Örnek sayımız çok az olduğu için küçük tuttuk
    per_device_eval_batch_size=16,
    num_train_epochs=3,                       # Modelin veriyi kaç tur döneceği (Epoch)
    weight_decay=0.01,
    eval_strategy="epoch",                    # Her epoch sonunda doğruluğu test et
    save_strategy="epoch",
    load_best_model_at_end=True,              # En başarılı modeli hafızada tut
    logging_steps=1,                          # Logları hemen görmek için 1 yaptık
    report_to="none"                          # Harici raporlama araçlarını kapat
)

# 5. Trainer (Eğitici) Kurulumu
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

# 6. Eğitimi Başlatıyoruz
print("\n--- Model Eğitimi Başlıyor ---")
trainer.train()

# 7. Modeli ve Tokenizer'ı Yerel Klasöre Kaydetme
drive_kayit_yolu = "/content/drive/MyDrive/en_iyi_detektor_modeli"

trainer.save_model(drive_kayit_yolu)
tokenizer.save_pretrained(drive_kayit_yolu)
print(f"\nEğitim tamamlandı ve model Google Drive'ınıza ({drive_kayit_yolu}) başarıyla kaydedildi!")
#trainer.save_model("./en_iyi_detektor_modeli")
#tokenizer.save_pretrained("./en_iyi_detektor_modeli")
#print("\nEğitim tamamlandı ve model './en_iyi_detektor_modeli' klasörüne kaydedildi!")

In [ ]:
import torch
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ==========================================
# ⚙️ MODELİ SAKİNLEŞTİRME AYARLARI
# ==========================================
AI_ESIK = 70.0     # Modelin AI demesi için çıtayı iyice yukarı çekiyoruz.
SICAKLIK = 3.5     # Değer büyüdükçe (örn: 2.0 - 3.5) modelin %99'luk inatçılığı kırılır,
                   # yüzdeler daha insani seviyelere (%60-%70) çekilir.

model_path = "/content/drive/MyDrive/en_iyi_detektor_modeli"

if not os.path.exists("/content/drive"):
    print("Drive bağlı değil, bağlanılıyor...")
    from google.colab import drive
    drive.mount('/content/drive')

if not os.path.exists(model_path):
    print(f"HATA: '{model_path}' bulunamadı!")
else:
    print("Model yükleniyor...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
    model.eval()
    print("Model başarıyla yüklendi!\n")

    def metni_analiz_et(metin, threshold=95.0, temperature=2.5):
        inputs = tokenizer(metin, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        # 🎯 HİLE BURADA: Ham puanları (logits) sıcaklık değerine bölerek yumuşatıyoruz.
        yumusak_logits = outputs.logits / temperature
        olasiliklar = torch.softmax(yumusak_logits, dim=-1)[0]

        insan_skoru = olasiliklar[0].item() * 100
        yz_skoru = olasiliklar[1].item() * 100

        # Karar mekanizması yeni eşiğe göre çalışıyor
        if yz_skoru >= threshold:
            karar = "Yapay Zeka Tarafından Yazılmış 🤖"
        else:
            karar = "İnsan Tarafından Yazılmış 👨‍💻"

        return karar, insan_skoru, yz_skoru

    print("=== YAPAY ZEKA METİN DETEKTÖRÜ (GELİŞMİŞ MOD) ===")
    print(f"Hassasiyet Barajı: %{AI_ESIK} | Yumuşatma Katsayısı: {SICAKLIK}")
    print("Çıkış yapmak için 'q' yazın.")

    while True:
        kullanici_metni = input("\nAnaliz edilecek metni girin:\n> ")

        if kullanici_metni.strip().lower() == 'q':
            print("Program kapatıldı.")
            break

        if not kullanici_metni.strip():
            continue

        karar, insan_yuzde, yz_yuzde = metni_analiz_et(kullanici_metni, threshold=AI_ESIK, temperature=SICAKLIK)

        print("\n" + "="*45)
        print(f"📊 KARAR: {karar}")
        print(f"👨‍💻 Dengelenmiş İnsan Tahmini: %{insan_yuzde:.2f}")
        print(f"🤖 Dengelenmiş Yapay Zeka Tahmini: %{yz_yuzde:.2f}")
        print("="*45)

